# Stock Market Price Prediction & Intelligent Trading Signal System
### College Capstone / PBL Machine Learning Project Experimentation Notebook

This notebook demonstrates the experimental progression through all 5 modules of the pipeline:
1. **Module 1:** Data Collection (yfinance)
2. **Module 2:** Data Preprocessing (Cleaning & Chronological Ordering)
3. **Module 3:** Feature Engineering (Lags, Technical Indicators, Target Construction)
4. **Module 4:** Machine Learning Training (Linear Regression, Random Forest, XGBoost)
5. **Module 5:** Evaluation, Trading Signal Generation & Risk Analysis

In [ ]:
import sys
import os
# Add parent directory to path to enable modular imports
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Module 1: Data Collection
Downloading historical OHLCV data for Apple Inc. (`AAPL`).

In [ ]:
from src.data_collection import download_stock_data

ticker = "AAPL"
start_date = "2020-01-01"
end_date = "2025-01-01"

raw_df, msg = download_stock_data(ticker, start_date, end_date)
print(msg)
raw_df.head()

## 2. Module 2: Data Preprocessing
Sanitize data, enforce strictly chronological ordering, and validate positive price constraints.

In [ ]:
from src.preprocessing import preprocess_data

clean_df, report = preprocess_data(raw_df, ticker=ticker)
print("Preprocessing Report:", report)
clean_df.info()

## 3. Module 3: Feature Engineering
Construct $t-1$ lag features, SMA 20, SMA 50, EMA 20, RSI 14, MACD, and 20-day volatility.
Define Target as $Close_{t+1}$ using `shift(-1)` without any forward leakage.

In [ ]:
from src.feature_engineering import build_features, TARGET_COLUMN

features_df, feature_cols = build_features(clean_df, include_target=True)
print(f"Total engineered features: {len(feature_cols)}")
print("Feature Columns:", feature_cols)
features_df[['Date', 'Close', 'Prev_Close', 'SMA_20', 'RSI_14', TARGET_COLUMN]].head()

## 4. Module 4: Machine Learning Model Training
Perform chronological 80/20 train/test split. Train Linear Regression, Random Forest, and XGBoost.

In [ ]:
from src.model_training import train_models

train_results = train_models(features_df, feature_cols, TARGET_COLUMN, ticker=ticker, test_ratio=0.20)
print(f"Training set samples: {len(train_results['train_df'])}")
print(f"Testing set samples:  {len(train_results['test_df'])}")
print("Trained Models:", list(train_results['models'].keys()))

## 5. Module 5: Evaluation & Model Comparison
Compare models across MAE, RMSE, $R^2$, and Directional Accuracy.

In [ ]:
from src.evaluation import compare_models

comp_df, best_model = compare_models(
    train_results['predictions'], 
    train_results['test_actual'], 
    current_close=train_results['test_df']['Close']
)
print(f"Best performing model: {best_model}")
comp_df

## 6. Trading Signal & Risk Analysis
Generate next-day inference, compute Expected Return %, assign indicative BUY/HOLD/SELL signal, and classify market risk.

In [ ]:
from src.feature_engineering import get_latest_inference_features
from src.prediction import predict_next_day, get_feature_importance
from src.signals import build_signal_summary

latest_feat, latest_meta = get_latest_inference_features(clean_df)
active_model = train_results['models'][best_model]

pred_price = predict_next_day(active_model, best_model, train_results['scaler'], latest_feat)
current_price = latest_meta['current_close']

signal_summary = build_signal_summary(
    current_price=current_price,
    predicted_price=pred_price,
    annualized_volatility=latest_meta['volatility_20'],
    buy_threshold=1.5,
    sell_threshold=-1.5
)

for key, val in signal_summary.items():
    print(f"{key.upper():<20}: {val}")

## 7. Feature Importance Analysis

In [ ]:
imp_df = get_feature_importance(active_model, best_model, feature_cols)
imp_df.head(8)